# 02 - Cleaning Data – Tai nghe / Headphone

Notebook này tách riêng phần **làm sạch dữ liệu** cho đề tài tai nghe:

- Đọc dữ liệu gộp từ `raw_data/raw_data.csv` (sau khi chạy `scripts/merge_raw_to_raw_data.py`).
- Khảo sát sơ bộ: `info()`, missing.
- Làm sạch: loại trùng, chuẩn hoá `price_vnd`, các cột nhị phân, chuẩn hoá `connection`, `battery_life_hours`, `weight_gram`.
- Lưu kết quả ra `clean_data/headphone_clean.csv` để dùng cho notebook phân tích.


## 1. Import thư viện & đọc dữ liệu raw


In [1]:
import os
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.0f}".format

PROJECT_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
RAW_CSV = os.path.join(PROJECT_DIR, "raw_data", "raw_data.csv")
CLEAN_DIR = os.path.join(PROJECT_DIR, "clean_data")
os.makedirs(CLEAN_DIR, exist_ok=True)

df = pd.read_csv(RAW_CSV, encoding="utf-8")
print("File raw:", RAW_CSV)
print("Số dòng raw ban đầu:", len(df))
df.head()


File raw: c:\Study\KHDL\GK\de_tai_2_headphone\raw_data\raw_data.csv
Số dòng raw ban đầu: 1029


,source,url,name,price_raw,price_vnd,brand,type,is_gaming,is_wireless,has_mic,connection,battery_life_hours,weight_gram
0,cellphones,https://cellphones.com.vn/tai-nghe-chup-tai-so...,Tai nghe Bluetooth chụp tai Sony WH-1000XM6,11.990.000đ,11990000,Sony,over-ear,0,1,0,3.5mm,40,254.0
1,cellphones,https://cellphones.com.vn/tai-nghe-chup-tai-se...,Tai nghe chụp tai Sennheiser HDB 630,14.900.000đ,14900000,Sennheiser,over-ear,0,0,0,NaN,60,NaN
2,cellphones,https://cellphones.com.vn/tai-nghe-chup-tai-so...,Tai nghe Bluetooth chụp tai Sony WH-1000XM5,7.990.000đ,7990000,Sony,over-ear,0,1,0,3.5mm,NaN,40.0
3,cellphones,https://cellphones.com.vn/tai-nghe-chup-tai-so...,Tai nghe Bluetooth chụp tai Sony WH-CH520,1.290.000đ,1290000,Sony,over-ear,0,1,0,NaN,50,40.0
4,cellphones,https://cellphones.com.vn/tai-nghe-chup-tai-jb...,Tai nghe Bluetooth chụp tai JBL Tune 520BT,1.360.000đ,1360000,JBL,over-ear,0,1,0,Type C,57,157.0


## 2. Khảo sát sơ bộ dữ liệu


In [2]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1029 entries, 0 to 1028
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   source              1029 non-null   object 
 1   url                 1029 non-null   object 
 2   name                1029 non-null   object 
 3   price_raw           1024 non-null   object 
 4   price_vnd           1029 non-null   int64  
 5   brand               1026 non-null   object 
 6   type                269 non-null    object 
 7   is_gaming           1026 non-null   float64
 8   is_wireless         1026 non-null   float64
 9   has_mic             1026 non-null   float64
 10  connection          428 non-null    object 
 11  battery_life_hours  339 non-null    float64
 12  weight_gram         346 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 104.6+ KB


In [3]:
missing = df.isna().mean().sort_values(ascending=False)
missing


type                 1
battery_life_hours   1
weight_gram          1
connection           1
price_raw            0
is_wireless          0
is_gaming            0
brand                0
has_mic              0
price_vnd            0
name                 0
url                  0
source               0
dtype: float64

## 2.1 Phân tích missing theo nhóm cột

Để hiểu rõ hơn chất lượng dữ liệu, ta tách missing thành các nhóm chính:
- **Nhóm giá**: `price_vnd`.
- **Nhóm nhận diện sản phẩm**: `brand`, `type`.
- **Nhóm thông số kỹ thuật (spec)**: `connection`, `battery_life_hours`, `weight_gram`.


In [4]:
# Tỷ lệ thiếu theo nhóm biến
cols_price = ['price_vnd']
cols_id = [c for c in ['brand', 'type'] if c in df.columns]
cols_spec = [c for c in ['connection', 'battery_life_hours', 'weight_gram'] if c in df.columns]

summary = []
for name, cols in [('price', cols_price), ('id', cols_id), ('spec', cols_spec)]:
    for c in cols:
        miss = df[c].isna().mean()
        summary.append({'group': name, 'column': c, 'missing_ratio': miss})

missing_groups = pd.DataFrame(summary).sort_values(['group', 'missing_ratio'], ascending=[True, False])
missing_groups

,group,column,missing_ratio
2,id,type,1
1,id,brand,0
0,price,price_vnd,0
4,spec,battery_life_hours,1
5,spec,weight_gram,1
3,spec,connection,1


## 2.2 Loại bỏ các dòng thiếu thông tin quan trọng

Trong đề tài này, để một bản ghi **có ý nghĩa** cho mô hình và phân tích, ta yêu cầu:
- Có `price_vnd` (Y) – không có giá thì không thể dùng.
- Có ít nhất **một trong hai**: `brand` hoặc `type` để biết sản phẩm thuộc nhóm nào.

Các bản ghi thiếu đồng thời `price_vnd` hoặc cả `brand` & `type` sẽ bị loại bỏ.

In [5]:
# Loại bỏ dòng thiếu price_vnd hoặc thiếu cả brand & type
before_crit = len(df)

mask_keep = df['price_vnd'].notna()
if 'brand' in df.columns or 'type' in df.columns:
    has_brand_or_type = (
        (df.get('brand').notna() if 'brand' in df.columns else False) |
        (df.get('type').notna() if 'type' in df.columns else False)
    )
    mask_keep &= has_brand_or_type

df = df[mask_keep].copy()
print('Số dòng sau khi loại các bản ghi thiếu thông tin quan trọng:', len(df), '(bớt', before_crit - len(df), ')')

Số dòng sau khi loại các bản ghi thiếu thông tin quan trọng: 1026 (bớt 3 )


## 3. Làm sạch cơ bản

- Loại trùng theo `url`.
- Chuẩn hoá `price_vnd`.
- Ép kiểu các cột nhị phân (`is_gaming`, `is_wireless`, `has_mic`).


In [6]:
# Loại trùng theo URL
before = len(df)
df = df.drop_duplicates(subset=["url"]).copy()
print("Số dòng sau khi drop trùng url:", len(df), "(bớt", before - len(df), ")")

# Giá VND về numeric
df["price_vnd"] = pd.to_numeric(df["price_vnd"], errors="coerce")
print("Số dòng thiếu price_vnd sau chuyển kiểu:", df["price_vnd"].isna().sum())

# Các cột nhị phân
for col in ["is_gaming", "is_wireless", "has_mic"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)


Số dòng sau khi drop trùng url: 1026 (bớt 0 )
Số dòng thiếu price_vnd sau chuyển kiểu: 0


## 4. Chuẩn hoá cột `connection`

Loại bỏ text mô tả dài và gom nhóm về một số giá trị chuẩn.


In [7]:
valid_connections = ["3.5mm", "Type C", "USB-A", "USB-C", "Bluetooth", "Wireless"]

def normalize_connection(x: str):
    if not isinstance(x, str) or not x.strip():
        return np.nan
    s = x.strip()
    if len(s) > 20:
        return np.nan
    t = s.lower()
    if "3.5" in t:
        return "3.5mm"
    if "type c" in t or "usb-c" in t or "usb c" in t:
        return "Type C"
    if t.startswith("usb") and "type c" not in t:
        return "USB-A"
    if "bluetooth" in t or "wireless" in t:
        return "Bluetooth"
    return s if s in valid_connections else np.nan

if "connection" in df.columns:
    before_sample = df["connection"].value_counts().head(10)
    df["connection"] = df["connection"].apply(normalize_connection)
    after_sample = df["connection"].value_counts(dropna=False).head(10)
    print("Trước khi chuẩn hoá (top 10):")
    print(before_sample)
    print("\nSau khi chuẩn hoá (top 10):")
    print(after_sample)


Trước khi chuẩn hoá (top 10):
connection
3.5mm                                                           161
Type C                                                           46
Copyright © 2026                                                 41
Có dây                                                           12
USB                                                               7
Cách chỉnh độ sáng laptop Acer nhanh, đơn giản chi tiết 2026      7
PC                                                                4
LIGHTSPEED 2.4GHz                                                 3
2 x cổng USB 3.1                                                  3
Wireless/ Bluetooth                                               3
Name: count, dtype: int64

Sau khi chuẩn hoá (top 10):
connection
NaN          795
3.5mm        167
Type C        49
Bluetooth      8
USB-A          7
Name: count, dtype: int64


## 5. Chuẩn hoá `battery_life_hours`, `weight_gram`


In [ ]:
if "battery_life_hours" in df.columns:
    df["battery_life_hours"] = pd.to_numeric(df["battery_life_hours"], errors="coerce")
if "weight_gram" in df.columns:
    df["weight_gram"] = pd.to_numeric(df["weight_gram"], errors="coerce")

df[[c for c in ["price_vnd", "battery_life_hours", "weight_gram"] if c in df.columns]].describe()


,price_vnd,battery_life_hours,weight_gram
count,"1,026",339,344
mean,"3,838,804",23,105
std,"8,711,232",29,156
min,0,2,0
25%,"650,000",7,5
50%,"1,750,000",9,17
75%,"5,000,000",30,222
max,"168,990,000",240,"1,650"


## 6. Lưu dữ liệu đã clean


In [ ]:
out_path = os.path.join(CLEAN_DIR, "headphone_clean.csv")
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Số dòng sau clean:", len(df))
print("Đã lưu:", out_path)


Số dòng sau clean: 1026
Đã lưu: c:\Study\KHDL\GK\de_tai_2_headphone\clean_data\headphone_clean.csv
